In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np
from pathlib import Path
import glob
import h5py

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from skewerjax.simulations.simulations import TNGSimulation

In [ ]:
base_path = "/home/STORAGE/TNG50-3"
snapnum = 25

sim = TNGSimulation(base_path)

In [ ]:
# --------------------------------------------
# 1) High-level summary
# --------------------------------------------
print("\n=== Simulation summary ===")
print(sim.summary())

print("\n=== Snapshot numbers discovered ===")
print(sim.snapnums)

print("\n=== Common (stable) cosmology/metadata ===")
for k, v in sim.cosmo.items():
    if v is None:
        continue
    print(f"{k:24s}: {v}")

# --------------------------------------------
# 2) Inspect a particular snapshot
# --------------------------------------------
snapnum = sim.snapnums[len(sim.snapnums)//2]  # pick a middle snapshot, or set snapnum=25
info = sim.get(snapnum)

print(f"\n=== Snapshot {snapnum:03d} basic info ===")
print("snapdir:", info.snapdir)
print("nfiles :", len(info.files))
print("files  :")
for p in info.files[:10]:
    print("  ", p.name)
if len(info.files) > 10:
    print("  ...")

# --------------------------------------------
# 3) Cosmology dictionary for that snapshot
# --------------------------------------------
print(f"\n=== Snapshot {snapnum:03d} parsed cosmology ===")
for k, v in info.cosmo.items():
    if v is None:
        continue
    if isinstance(v, np.ndarray):
        v = v.tolist()
    print(f"{k:24s}: {v}")

# --------------------------------------------
# 4) Raw metadata keys (Header / Parameters / Config)
# --------------------------------------------
print(f"\n=== Snapshot {snapnum:03d} Header keys ===")
print(sorted(info.header.keys()))

print(f"\n=== Snapshot {snapnum:03d} Parameters keys ===")
print(sorted(info.parameters.keys()) if info.parameters else "(none)")

print(f"\n=== Snapshot {snapnum:03d} Config keys ===")
print(sorted(info.config.keys()) if info.config else "(none)")

# Optionally print specific raw header attrs commonly useful
want_header = [
    "BoxSize", "Redshift", "Time", "HubbleParam",
    "Omega0", "OmegaLambda", "OmegaBaryon",
    "NumFilesPerSnapshot", "NumPart_Total", "MassTable",
]
print(f"\n=== Snapshot {snapnum:03d} selected Header values (if present) ===")
for k in want_header:
    if k in info.header:
        v = info.header[k]
        if isinstance(v, np.ndarray):
            v = v.tolist()
        print(f"{k:24s}: {v}")

# --------------------------------------------
# 5) Loop over all snapshots: (snapnum, z, a) quick table
# --------------------------------------------
print("\n=== All snapshots: z and a ===")
print(f"{'snap':>6} {'z':>12} {'a':>12} {'nfiles':>8}")
for n in sim.snapnums:
    c = sim.get(n).cosmo
    z = c.get("Redshift", None)
    a = c.get("ScaleFactor", None)
    nf = len(sim.get(n).files)
    print(f"{n:6d} {str(z):>12} {str(a):>12} {nf:8d}")

In [ ]:
from scipy.spatial import cKDTree

In [ ]:
# sample the points in a uniform grid with n_grid points in each dimension
grid = 16
print("grid size:", grid**3)
x = np.linspace(0, sim.cosmo["BoxSize"], grid, dtype=np.float32)
X, Y, Z = np.meshgrid(x, x, x, indexing="ij")
sample_points_ckpc_over_h = np.vstack([X.ravel(), Y.ravel(), Z.ravel()]).T

In [ ]:
# choose what you want to read (edit this list)
datasets = [
    "Coordinates",
    "Velocities",
    "InternalEnergy",
    "ElectronAbundance",
    "GFM_Metals",
    "NeutralHydrogenAbundance",
    "StarFormationRate",
]


info = sim.get(snapnum)

L_box_ckpc_over_h = float(info.cosmo["BoxSize"])  # ckpc/h

sample_points_ckpc_over_h = np.asarray(sample_points_ckpc_over_h, dtype=np.float32)
if sample_points_ckpc_over_h.shape[1] != 3:
    raise ValueError("sample_points_ckpc_over_h must have shape (B,3)")
sample_points_ckpc_over_h[sample_points_ckpc_over_h == L_box_ckpc_over_h] = 0.0
assert np.all(sample_points_ckpc_over_h < L_box_ckpc_over_h)

for ii, file_path in enumerate(info.files):

    print(f"\n=== file {ii} ===")

    if sample_points_ckpc_over_h is None:
        with h5py.File(file_path, "r") as f:
            Ni = f["PartType0"]["Coordinates"].shape[0]
        neighbors_by_file = np.arange(Ni, dtype=np.int64)

    else:
        with h5py.File(file_path, "r") as f:
            dens = f["PartType0"].get("Density", None)[()].astype(np.float32, copy=False)
            mass = f["PartType0"].get("Masses", None)[()].astype(np.float32, copy=False)

            coords = f["PartType0"].get("Coordinates", None)[()].astype(np.float32, copy=False)

        Rmax_ckpc_over_h = np.max(np.cbrt(mass / dens))
        coords[coords == L_box_ckpc_over_h] = 0.0

        tree = cKDTree(coords, boxsize=L_box_ckpc_over_h)

        cand_lists = tree.query_ball_point(sample_points_ckpc_over_h, r=Rmax_ckpc_over_h)

        unique_indices = np.unique(np.concatenate(cand_lists).astype(np.int64))

        print(f"unique_indices = ", len(unique_indices))


        # with h5py.File(file_path, "r") as f:
        #     for name in datasets:
        #         arr = f["PartType0"].get(name, None)[()].astype(np.float32, copy=False)
        #         print(f"   {name:24s} shape={arr.shape}")

In [ ]:
data = sim.load_snapdir_hdf5(
    snapnum=25,
    datasets=["Coordinates", "Density", "Masses"]
)

In [ ]:
print(data["data"].keys())
print(data["data"][0].keys())
print(data["data"][0]["Coordinates"].shape)

In [ ]:
selected = {
    0: np.array([0, 5, 10]),
    1: np.array([2, 3]),
    5: np.array([], dtype=int)
}

masked = sim.load_snapdir_hdf5_masked(
    snapnum=25,
    datasets=["Coordinates"],
    file_indices=[0,1,5],
    selected_indices_by_file=selected
)

print(masked["data"][0]["Coordinates"].shape)


In [ ]:
data = sim.load_snapdir_hdf5(
    snapnum=snapnum, datasets=["Density", "Masses"]
)

R_eff_ckpc_over_h = []
for ii, file_id in enumerate(data['data'].keys()):
    R_eff_ckpc_over_h.append(np.cbrt(data['data'][file_id]['Masses'] / data['data'][file_id]['Density']))
R_eff_ckpc_over_h = np.concatenate(R_eff_ckpc_over_h, axis=0)

Rmax_ckpc_over_h = np.max(R_eff_ckpc_over_h)

print("Rmax_ckpc_over_h:", Rmax_ckpc_over_h)
print("box size ckpc_over_h:", sim.cosmo["BoxSize"])

In [ ]:
neighbours_by_file = sim.get_index_cKD_neighbours(
    sample_points=sample_points_ckpc_over_h, snapnum=snapnum, Rmax_ckpc_over_h=Rmax_ckpc_over_h,
)

In [ ]:
# Quick sanity check
tot = sum(len(v) for v in neighbours_by_file.values())
print("Total unique candidate cells:", tot)
print("Per file:", {fid: len(v) for fid, v in neighbours_by_file.items()})

In [ ]:
masked = sim.load_snapdir_hdf5_masked(
    snapnum=snapnum, datasets=["Coordinates", "Masses"], selected_indices_by_file = neighbours_by_file
)

In [ ]:
print(masked.keys())
print(masked["data"].keys())
print(masked["data"][0].keys())
print(masked["data"][0]["Coordinates"].shape)

In [ ]:
snapnum = 25
ions_to_compute=[("H", 1), ("Si", 2), ("Si", 3)]
fields_from_sim = sim.load_fields_from_simulation(
    snapnum=snapnum, sample_points_Mpc=sample_points_Mpc, ions_to_compute=ions_to_compute, HI_method="Rahmati"
)

In [ ]:
fields_from_sim[snapnum]

In [ ]:
print(fields_from_sim[snapnum]["coordinates_cMpc"].shape)
print(fields_from_sim[snapnum]["number_density_cm3"]["gas"].shape)
print(fields_from_sim[snapnum]["velocity_kms"].shape)

In [ ]:
print(fields_from_sim[snapnum]["radius_cMpc"])

In [ ]:
from matplotlib.collections import EllipseCollection

# --- Existing inputs ---
coords = np.asarray(fields_from_sim[snapnum]["coordinates_cMpc"])                # (N,3) cMpc
radius = np.asarray(fields_from_sim[snapnum]["radius_cMpc"])                     # (N,) cMpc

ngas   = np.asarray(fields_from_sim[snapnum]["number_density_cm3"]["gas"])      # (N,) cm^-3
Tgas   = np.asarray(fields_from_sim[snapnum]["temperature_K"])           # (N,) K
vpec   = np.asarray(fields_from_sim[snapnum]["velocity_kms"])            # (N,3) km/s

# --------------------------------------------
# Choose a thin slice: y ~ median(y)
# --------------------------------------------
y = coords[:, 1]
y0 = np.median(y)
dy = 0.02 * (y.max() - y.min())   # 2% thickness
m = np.abs(y - y0) < dy

xs = coords[m, 0]
zs = coords[m, 2]

n_s    = ngas[m]
T_s    = Tgas[m]
v_s    = vpec[m]
vmag_s = np.sqrt((v_s**2).sum(axis=1))
radius_s = radius[m]

# --------------------------------------------
# Build simple 2D binned mean images (no scipy)
# --------------------------------------------
def binned_mean(x, y, val, nx=n_grid, ny=n_grid):
    x0, x1 = x.min(), x.max()
    y0, y1 = y.min(), y.max()

    ix = np.floor((x - x0) / (x1 - x0) * nx).astype(int)
    iy = np.floor((y - y0) / (y1 - y0) * ny).astype(int)

    good = (ix >= 0) & (ix < nx) & (iy >= 0) & (iy < ny) & np.isfinite(val)
    ix, iy, val = ix[good], iy[good], val[good]

    s = np.zeros((ny, nx), dtype=float)
    c = np.zeros((ny, nx), dtype=int)

    np.add.at(s, (iy, ix), val)
    np.add.at(c, (iy, ix), 1)

    img = np.full((ny, nx), np.nan)
    ok = c > 0
    img[ok] = s[ok] / c[ok]

    extent = (x0, x1, y0, y1)
    return img, extent

# --------------------------------------------
# Make panel images
# --------------------------------------------
img_logng, extent = binned_mean(xs, zs, np.log10(np.clip(n_s,  1e-30, None)))
img_logT,  _      = binned_mean(xs, zs, np.log10(np.clip(T_s,  1e0,   None)))
img_vmag,  _      = binned_mean(xs, zs, vmag_s)

panels = [
    (img_logng, "Blues",   r"$\langle \log_{10} n_{\rm gas}\rangle\ [{\rm cm^{-3}}]$"),
    (img_logT,  "Oranges", r"$\langle \log_{10} T_{\rm gas}\rangle\ [{\rm K}]$"),
    (img_vmag,  "Greys",   r"$\langle |\vec v_{\rm pec}|\rangle\ [{\rm km/s}]$"),
]

fig, axes = plt.subplots(4, 1, figsize=(7, 7*0.65*4), sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()

# Quiver subsample
rng = np.random.default_rng(0)
nq = 3000
idx_q = rng.choice(xs.shape[0], size=min(nq, xs.shape[0]), replace=False)
vx_q = v_s[idx_q, 0]
vz_q = v_s[idx_q, 2]

# --------------------------------------------
# scatter where marker size ~ log(volume)
axv = axes[0]

# Subsample to keep rendering reasonable (draws one ellipse per point)
rng = np.random.default_rng(0)
nmax = 100000  # tune if needed
idx = rng.choice(xs.shape[0], size=min(nmax, xs.shape[0]), replace=False)

x_sc = xs[idx]
z_sc = zs[idx]
radius_sc = radius_s[idx]

# Convert to *diameters* for EllipseCollection
w = 2.0 * radius_sc  # cMpc
h = 2.0 * radius_sc  # cMpc

# Color by log10(volume) (or log10(radius), your call)
c = np.log10(np.clip(radius_sc, 1e-40, None))

# --- Add to an axis `ax` ---
# IMPORTANT: EllipseCollection sizes are in *data units* if units='xy' and transOffset=ax.transData
ec = EllipseCollection(
    widths=w,
    heights=h,
    angles=0.0,
    units="xy",
    offsets=np.column_stack([x_sc, z_sc]),
    transOffset=axv.transData,
    cmap="viridis",
    alpha=0.35,
    linewidths=0.0,
)
ec.set_array(c)
axv.add_collection(ec)

# Keep axes limits consistent (EllipseCollection does not auto-expand limits reliably)
axv.set_xlim(extent[0], extent[1])
axv.set_ylim(extent[2], extent[3])
axv.set_aspect("equal", adjustable="box")

cb = plt.colorbar(ec, ax=axv, pad=0.02, fraction=0.046)
cb.set_label(r"$\log_{10} V_{\rm cell}\ [{\rm cMpc^3}]$")




axv.set_ylabel(r"$z\ [{\rm cMpc}]$")
axv.set_aspect("equal", adjustable="box")

# 3 panels: binned means
for ax, (img, cmap, cblabel) in zip(axes[1:], panels):
    vmin = np.nanpercentile(img, 2)
    vmax = np.nanpercentile(img, 98)

    im = ax.imshow(
        img, origin="lower", extent=extent, aspect="auto",
        cmap=cmap, vmin=vmin, vmax=vmax
    )
    cb = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.046)
    cb.set_label(cblabel)

    ax.set_ylabel(r"$z\ [{\rm cMpc}]$")
    ax.set_aspect("equal", adjustable="box")

# velocity quiver (xz projection) on the 3rd panel (vmag panel)
axes[-1].quiver(
    xs[idx_q], zs[idx_q], vx_q, vz_q,
    color="red", angles="xy", scale_units="xy",
    scale=None, width=0.0035, alpha=0.55
)

axes[-1].set_xlabel(r"$x\ [{\rm cMpc}]$")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 4))
axes = axes.ravel()

axes[0].hist(np.log10(np.clip(ngas, 1e-30, None)), bins=80)

axes[1].hist(np.log10(np.clip(Tgas, 1e-30, None)), bins=80)

vmag = np.sqrt((vpec**2).sum(axis=1))
axes[2].hist(np.log10(np.clip(vmag, 1e-10, None)), bins=80)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib import colors as mcolors

# ----------------------------
# Inputs
# ----------------------------
coords = np.asarray(fields_from_sim[snapnum]["coordinates_cMpc"])  # (N,3) cMpc
ndict  = fields_from_sim[snapnum]["number_density_cm3"]               # nested dict

# ----------------------------
# Slice: y ~ median(y)
# ----------------------------
y = coords[:, 1]
y0 = np.median(y)
dy = 0.02 * (y.max() - y.min())
m = np.abs(y - y0) < dy

xs = coords[m, 0]
zs = coords[m, 2]

# ----------------------------
# 2D binned mean
# ----------------------------
def binned_mean(x, y, val, nx=350, ny=350):
    x0, x1 = x.min(), x.max()
    y0, y1 = y.min(), y.max()

    ix = np.floor((x - x0) / (x1 - x0) * nx).astype(int)
    iy = np.floor((y - y0) / (y1 - y0) * ny).astype(int)

    good = (ix >= 0) & (ix < nx) & (iy >= 0) & (iy < ny) & np.isfinite(val)
    ix, iy, val = ix[good], iy[good], val[good]

    s = np.zeros((ny, nx), dtype=float)
    c = np.zeros((ny, nx), dtype=int)
    np.add.at(s, (iy, ix), val)
    np.add.at(c, (iy, ix), 1)

    img = np.full((ny, nx), np.nan)
    ok = c > 0
    img[ok] = s[ok] / c[ok]
    extent = (x0, x1, y0, y1)
    return img, extent

# ----------------------------
# Collect components to plot
# ----------------------------
components = []
for k, v in ndict.items():
    if isinstance(v, dict):
        for kk, arr in v.items():
            if arr is None:
                continue
            arr = np.asarray(arr)
            if arr.ndim != 1:
                continue
            label = f"{k}/{kk}"
            components.append((label, arr[m]))
    else:
        arr = np.asarray(v)
        if arr.ndim == 1:
            components.append((str(k), arr[m]))

def _sort_key(item):
    label = item[0]
    if label == "gas":
        return (0, label)
    if label.endswith("/all"):
        return (1, label)
    return (2, label)

components = sorted(components, key=_sort_key)

# ----------------------------
# OPTIONS (two independent toggles)
# ----------------------------
# 1) Colormaps:
#    - If USE_SHARED_CMAP=True: all panels use shared_cmap
#    - Else: each panel uses cmap_by_label.get(label, default_cmap)
USE_SHARED_CMAP = False

shared_cmap = "gist_stern"
default_cmap = "gist_stern"
cmap_by_label = {
    "gas": "Greys",
    "H/all": "Blues",
    "H/1": "Greens",
    "Si/all": "Reds",
    "Si/2": "Oranges",
    "Si/3": "Purples",
}

# 2) Normalization:
#    - If USE_COMMON_NORM=True: *one* vmin/vmax for all panels (even if colormaps differ)
#    - Else: each panel has its own vmin/vmax (recommended if panels span very different ranges)
USE_COMMON_NORM = False

# ----------------------------
# Build all panel images first
# ----------------------------
imgs, extents, labels = [], [], []
for label, n_s in components:
    img, extent = binned_mean(xs, zs, np.log10(np.clip(n_s, 1e-40, None)))
    imgs.append(img)
    extents.append(extent)
    labels.append(label)

# ----------------------------
# Normalization (common or per-panel)
# ----------------------------
norms = []
if USE_COMMON_NORM:
    all_vals = np.concatenate([
        img[np.isfinite(img)].ravel()
        for img in imgs
        if np.any(np.isfinite(img))
    ])
    vmin = np.nanpercentile(all_vals, 2) if all_vals.size else np.nan
    vmax = np.nanpercentile(all_vals, 98) if all_vals.size else np.nan
    common_norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    norms = [common_norm] * len(imgs)
else:
    vmin, vmax = None, None
    for img in imgs:
        vals = img[np.isfinite(img)]
        if vals.size == 0:
            norms.append(mcolors.Normalize(vmin=np.nan, vmax=np.nan))
            continue
        v0 = np.nanpercentile(vals, 2)
        v1 = np.nanpercentile(vals, 98)
        norms.append(mcolors.Normalize(vmin=v0, vmax=v1))

# ----------------------------
# Plot: single column, share axes, label top-right
# ----------------------------
n_panels = len(labels)
fig, axes = plt.subplots(
    n_panels, 1,
    figsize=(8.2, 3.8 * n_panels),
    sharex=True, sharey=True
)
axes = np.atleast_1d(axes).ravel()

for ax, label, img, extent, norm_i in zip(axes, labels, imgs, extents, norms):
    cmap_i = shared_cmap if USE_SHARED_CMAP else cmap_by_label.get(label, default_cmap)

    im = ax.imshow(
        img, origin="lower", extent=extent,
        cmap=cmap_i, norm=norm_i,
        aspect="equal"
    )

    cb = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.046)
    cb.set_label(r"$\langle \log_{10} n \rangle\ [{\rm cm^{-3}}]$")
    cb.ax.set_title(label, fontsize=10, pad=6)

    ax.text(
        0.98, 0.98, label,
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=12,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.75, edgecolor="none")
    )

    ax.set_ylabel(r"$z\ [{\rm cMpc}]$")
    ax.grid(color="w", linewidth=0.8, alpha=0.12)
    ax.set_aspect("equal", adjustable="box")

axes[-1].set_xlabel(r"$x\ [{\rm cMpc}]$")

plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# Inputs
# ----------------------------
coords = np.asarray(fields_from_sim[snapnum]["coordinates_cMpc"])  # (N,3) cMpc
ndict  = fields_from_sim[snapnum]["number_density_cm3"]               # nested dict

# ----------------------------
# Collect components (same logic as your panels)
# ----------------------------
components = []
for k, v in ndict.items():
    if isinstance(v, dict):
        for kk, arr in v.items():
            if arr is None:
                continue
            arr = np.asarray(arr)
            if arr.ndim != 1:
                continue
            label = f"{k}/{kk}"
            components.append((label, arr))
    else:
        arr = np.asarray(v)
        if arr.ndim == 1:
            components.append((str(k), arr))

def _sort_key(item):
    label = item[0]
    if label == "gas":
        return (0, label)
    if label.endswith("/all"):
        return (1, label)
    return (2, label)

components = sorted(components, key=_sort_key)

# ----------------------------
# Histogram options
# ----------------------------
BINS = 120
LOG_FLOOR = 1e-40  # avoid -inf in log10
DENSITY = False    # True -> normalized histograms

# ----------------------------
# Plot: one column of histograms, one per component
# ----------------------------
n_panels = len(components)
fig, axes = plt.subplots(n_panels, 1, figsize=(8.2, 2.8 * n_panels), sharex=True)
axes = np.atleast_1d(axes).ravel()

for ax, (label, arr) in zip(axes, components):
    x = np.log10(np.clip(arr, LOG_FLOOR, None))

    ax.hist(x, bins=BINS, density=DENSITY)
    ax.grid(alpha=0.25)
    ax.set_ylabel("count" if not DENSITY else "pdf")
    ax.text(
        0.98, 0.92, label,
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.8, edgecolor="none"),
    )
    ax.set_yscale("log")

axes[-1].set_xlabel(r"$\log_{10}\, n\ [{\rm cm^{-3}}]$")

plt.tight_layout()
plt.show()
